# Lesson 1.8: EDA Basic

Welcome to the Exploratory Data Analysis (EDA) lesson! This notebook takes a raw dataset and walks the
full path: understanding its structure, cleaning it, transforming it, and moving it in and out of files.

**Structure — the four learning outcomes, in order:**
* **Part 1: Descriptive Statistics** — *summarise* a dataset: shape, data types, distributions.
* **Part 2: Data Quality** — *handle* the messy reality: missing values, duplicates, outliers.
* **Part 3: Data Transformation** — *transform* for analysis: mapping, labels, strings, categoricals.
* **Part 4: Reading & Writing Data** — *read and write* CSV, Excel, databases, pickle.

**For Learners:** Read the comments in the code to understand exactly what each parameter does.

> **🧭 Today's flow — 150 minutes.** One messy dataset, four learning outcomes, in order:
>
> | | Section | Learning outcome | Time |
> |---|---|---|---|
> | — | Setup + why this matters | | 5 min |
> | **Part 1** | Descriptive Statistics | **Summarise** a dataset: shape, dtypes, distributions | 33 min |
> | ☕ | *Break* | | 10 min |
> | **Part 2** | Data Quality | **Handle** missing values, duplicates, outliers | 42 min |
> | ☕ | *Break* | | 10 min |
> | **Part 3** | Data Transformation | **Transform**: mapping, labels, strings, categories, dates, grouping | 35 min |
> | **Part 4** | Reading & Writing Data | **Read and write** CSV, JSON, Excel, databases | 15 min |
>
> **The spine:** we work on one file, `data/patients.csv`, from start to finish. Each section
> improves the same `clean` table, and Part 4 saves it. Small hand-built tables appear alongside
> it as *drills* — they isolate one method so you can see exactly what it does.
>
> Each of Parts 1–3 ends with a **🛠️ Group Exercise**. Deep dives live in `reference.md`;
> the Appendix at the end is self-study.

### Setup

Import the libraries, then load the dataset we will use all session.

In [830]:
# 👉 Load the two toolkits we need. `pd` and `np` are just short nicknames so we can
#    type `pd.something` instead of `pandas.something`. Run this cell first, every session.
import pandas as pd
import numpy as np

In [831]:
# 👉 Load the dataset we will use all session: 38 rows of (synthetic) hospital records.
#    `read_csv` reads a comma-separated text file into a DataFrame -- a table with named columns.
#    pandas already treats an empty field and the text "NA" as missing.
patients = pd.read_csv("../data/patients.csv")

patients

,patient_id,name,contact,admit_date,ward,age,blood_type,days_admitted,bill_sgd
0,P001,Aisha Rahman,aisha.rahman@example.com,2024-01-03,Cardiology,54,O+,5.0,4820.50
1,P002,Brandon Lee,brandon.lee@example.net,2024-01-04,ICU,67,A-,12.0,18450.00
2,P003,Chen Wei Ming,chen.weiming@example.com,2024-01-04,Orthopaedics,41,B+,3.0,2310.75
3,P004,Divya Nair,divya.nair@example.org,2024-01-05,icu,150,O-,9.0,15200.00
4,P005,Ethan Tan,NaN,2024-01-06,Cardiology,38,NaN,4.0,3990.25
5,P006,Farah Ismail,farah.ismail@example.com,2024-01-07,General,29,A+,2.0,1250.00
6,P007,gerald ong,gerald.ong@example.net,2024-01-08,I.C.U,72,B-,15.0,21870.40
7,P008,Hui Ling Goh,huiling.goh@example.com,2024-01-08,Orthopaedics,63,O+,NaN,5640.00
8,P009,Ismail Yusof,ismail.yusof@example.org,2024-01-09,General,45,AB+,3.0,1780.90
9,P010,Jasmine Koh,jasmine.koh@example.com,2024-01-10,Cardiology,-3,A+,6.0,4410.00


---

## Part 1: Descriptive Statistics — Summarising Your Data

**Learning outcome 1:** *Summarise a dataset using descriptive statistics and identify its shape, data types, and distributions.*

**Goal:** Before you clean anything, you must know what you have. First the five-move first-look
ritual you will use on every dataset for the rest of your career, then the summary methods
underneath it.

⏱️ 33 min (25 min taught + 8 min group exercise)

### 1.1: The First Look — a five-move ritual

**Before anything else: why bother?** Run the two cells below. This hospital has four wards and
no patient older than about 100. Look at what the data claims.

In [832]:
# 👉 `.value_counts()` counts how many rows have each value. How many wards do you count?
patients["ward"].value_counts()

General         10
Cardiology       9
ICU              7
Orthopaedics     7
icu              1
I.C.U            1
 ICU             1
orthopaedics     1
GENERAL          1
Name: ward, dtype: int64

In [833]:
# 👉 The oldest patient, and the youngest. Both are impossible.
patients["age"].max(), patients["age"].min()

(150, -3)

Nine wards for a four-ward hospital. A 150-year-old patient, and one aged -3.

Any average, chart, or model built on this table is wrong — and *nothing about the file warns you*.
Finding this before you analyse is the whole job of EDA. That is what today buys you.

---

**The ritual.** Five moves, always in this order, on every new dataset:

| Move | Question it answers |
|---|---|
| `.head()` | What do the rows actually look like? |
| `.shape` | How big is it? |
| `.info()` | What type is each column, and where are the holes? |
| `.dtypes` | Is anything stored as the wrong type? |
| `.describe()` | Are the numbers plausible? |

In [834]:
# 👉 Move 1: look at real rows. `.head()` shows the first 5 (pass a number for more).
#    Never analyse a table you have not looked at.
patients.head()

,patient_id,name,contact,admit_date,ward,age,blood_type,days_admitted,bill_sgd
0,P001,Aisha Rahman,aisha.rahman@example.com,2024-01-03,Cardiology,54,O+,5.0,4820.50
1,P002,Brandon Lee,brandon.lee@example.net,2024-01-04,ICU,67,A-,12.0,18450.00
2,P003,Chen Wei Ming,chen.weiming@example.com,2024-01-04,Orthopaedics,41,B+,3.0,2310.75
3,P004,Divya Nair,divya.nair@example.org,2024-01-05,icu,150,O-,9.0,15200.00
4,P005,Ethan Tan,NaN,2024-01-06,Cardiology,38,NaN,4.0,3990.25


In [837]:
# 👉 Move 2: how big? `.shape` gives (rows, columns). No brackets -- it is a value, not a method.
patients.shape

(38, 9)

In [838]:
# 👉 Move 3: the single most useful command in pandas. For every column it reports the
#    non-null count and the dtype. Any column whose count is below 38 has missing values.
patients.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 38 entries, 0 to 37
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   patient_id     38 non-null     object 
 1   name           38 non-null     object 
 2   contact        35 non-null     object 
 3   admit_date     38 non-null     object 
 4   ward           38 non-null     object 
 5   age            38 non-null     int64  
 6   blood_type     33 non-null     object 
 7   days_admitted  35 non-null     float64
 8   bill_sgd       38 non-null     float64
dtypes: float64(2), int64(1), object(6)
memory usage: 2.8+ KB


In [839]:
# 👉 Move 4: just the types. `object` means text (or mixed). Note `admit_date` is text,
#    not a date -- pandas will not guess that for you. We convert it in Part 3.
patients.dtypes

patient_id        object
name              object
contact           object
admit_date        object
ward              object
age                int64
blood_type        object
days_admitted    float64
bill_sgd         float64
dtype: object

In [840]:
# 👉 Move 5: the numbers. Read the min and max rows first: that is where impossible
#    values hide. `count` also tells you how many values were present per column.
patients.describe()

,age,days_admitted,bill_sgd
count,38.000000,35.000000,38.000000
mean,52.421053,-22.485714,7327.030263
std,23.482750,169.969266,7346.614406
min,-3.000000,-999.000000,-999.000000
25%,38.250000,3.000000,1830.800000
50%,51.000000,5.000000,4342.500000
75%,65.750000,8.500000,13145.000000
max,150.000000,16.000000,23100.000000


In [841]:
# 👉 `.describe()` skips text columns by default. Ask for them explicitly and you get
#    count / unique / top / freq instead -- useful for spotting spelling variants.
patients.describe(include="object")

,patient_id,name,contact,admit_date,ward,blood_type
count,38,38,35,38,38,33
unique,36,36,33,33,9,8
top,P021,Umar Farouk,umar.farouk@example.com,2024-01-12,General,O+
freq,2,2,2,2,10,8


**Write down what the ritual found** — this is our to-do list for Part 2:

1. **Missing values** — `contact` (3), `blood_type` (5), `days_admitted` (3).
2. **Duplicates** — 38 rows but only 36 patient IDs.
3. **Impossible values** — `age` 150 and -3; `days_admitted` and `bill_sgd` contain -999.
4. **Wrong type** — `admit_date` is text, not a date.
5. **Inconsistent categories** — 9 spellings of 4 wards.

### 1.2a: Those Statistics, Unpacked on Our Data

`.describe()` is a bundle of simpler methods. Here we take them one at a time — on a slice of
`patients` small enough that you can check every number in your head, but real enough that each
answer means something.

In [842]:
# 👉 `.loc[:7, [...]]` takes rows 0 to 7 and just two numeric columns. Small on purpose:
#    8 rows you can add up by hand. Real on purpose: every number is a patient.
sample = patients.loc[:7, ["age", "days_admitted"]]

sample

,age,days_admitted
0,54,5.0
1,67,12.0
2,41,3.0
3,150,9.0
4,38,4.0
5,29,2.0
6,72,15.0
7,63,NaN


**Reductions** — a whole column in, one number out. By default they work *down* the rows.

Note row 7: `days_admitted` is missing. Keep an eye on it.

In [843]:
# 👉 Total years of age, and total bed-days, across these 8 patients.
#    One number per column. Add the second column up by hand and check pandas agrees.
sample.sum()

age              514.0
days_admitted     50.0
dtype: float64

In [846]:
# 👉 The averages. `.mean()` divides by the number of values *present*, not by 8 --
#    which is why the two columns are divided by different counts. `.count()` proves it.
print(sample.mean())
print()
print(sample.count())

age              64.250000
days_admitted     7.142857
dtype: float64

age              8
days_admitted    7
dtype: int64


**`skipna` — what to do about the hole.** By default pandas ignores missing values and
carries on. Sometimes you want the opposite: a loud `NaN` that tells you the input was incomplete.

In [ ]:
# 👉 `skipna=False` refuses to silently work around the hole. `days_admitted` becomes NaN;
#    `age` is unaffected because it has no holes.
#    Which behaviour do you want? "Average stay = 7.14 days" hides that one stay is unknown.
sample.sum(skipna=False)

age              514.0
days_admitted      NaN
dtype: float64

**Indirect Statistics:** Finding *where* the max or min value is located (the index label).

In [847]:
# 👉 Not the biggest *value* -- the row *label* where it sits. Combine it with `.loc[...]`
#    and you can pull out the whole record: "which patient stayed the longest?"
longest = sample["days_admitted"].idxmax()

print("row label:", longest)
patients.loc[longest, ["patient_id", "name", "ward", "days_admitted"]]

row label: 6


patient_id                 P007
name               gerald ong  
ward                      I.C.U
days_admitted              15.0
Name: 6, dtype: object

In [848]:
# 👉 Same for the minimum, on both columns at once. Note `age` points at row 5 --
#    the youngest of these eight patients.
sample.idxmin()

age              5
days_admitted    5
dtype: int64

**Accumulations:** Computing cumulative sums.

In [849]:
# 👉 A running total: each row is itself plus everything above it. Read the bottom row of
#    `days_admitted` as "total bed-days used by the first 8 patients". The NaN stays NaN and
#    everything after it is affected -- another reason to clean before you calculate.
sample.cumsum()

,age,days_admitted
0,54,5.0
1,121,17.0
2,162,20.0
3,312,29.0
4,350,33.0
5,379,35.0
6,451,50.0
7,514,NaN


**`.describe()` — the bundle.** Everything above, in one call. You met it as move 5 of the
ritual; now you know what each row of its output actually is.

In [850]:
# 👉 All of the above in one call, plus the quartiles. This is why `.describe()` is move 5
#    of the ritual: it is the whole bundle. Look at max age -- 150. We fix that in Part 2.
sample.describe()

,age,days_admitted
count,8.000000,7.000000
mean,64.250000,7.142857
std,37.795502,4.947342
min,29.000000,2.000000
25%,40.250000,3.500000
50%,58.500000,5.000000
75%,68.250000,10.500000
max,150.000000,15.000000


**Categorical/Non-Numeric Data:** `.describe()` behaves differently for string data, showing counts and uniqueness.

In [851]:
# 👉 A Series is a single column of data. This one holds text, so `.describe()` switches to
#    text-style stats: how many, how many different ones, and which appears most often.
obj = pd.Series(["c", "a", "d", "a", "b", "b", "c", "c"])

# For object data, we get count, unique, top (most frequent), and freq
obj.describe()

count     8
unique    4
top       c
freq      3
dtype: object

In [852]:
# 👉 List the distinct values, in the order they first appear. Duplicates are dropped.
# Get unique values
obj.unique()

array(['c', 'a', 'd', 'b'], dtype=object)

In [853]:
# 👉 Count how many times each distinct value appears, most frequent first.
# Get frequency counts of each unique value
obj.value_counts()

c    3
a    2
b    2
d    1
dtype: int64

### 1.2b: The `axis` Sandbox — a deliberately meaningless table

One piece of notation left: **`axis`**. Every reduction can run *down* the rows (the default) or
*across* the columns.

Across-the-columns is hard to demonstrate on real data, because adding a patient's age to their
bed-days is nonsense. So we practise on a 4×2 grid of numbers that mean nothing at all — then you
will recognise the notation when you meet it in someone else's code.

In [854]:
# 👉 Build a small table by hand. The outer [ ] is a list of rows; each inner [ ] is one row.
#    `np.nan` is how Python/pandas writes 'this value is missing'. `index=` names the rows,
#    `columns=` names the columns. Last line is just `demo`, which tells Jupyter to display it.
demo = pd.DataFrame([[1.4, np.nan], [7.1, -4.5], [np.nan, np.nan], [0.75, -1.3]],
                  index=["a", "b", "c", "d"], columns=["one", "two"])

demo

,one,two
a,1.40,NaN
b,7.10,-4.5
c,NaN,NaN
d,0.75,-1.3


In [855]:
# 👉 Add up each column, top to bottom. One number comes back per column.
# Sums down the rows (returns sum for each column)
demo.sum()

one    9.25
two   -5.80
dtype: float64

In [856]:
# 👉 Same addition, but sideways: add across each row. `axis="columns"` means 'go across'.
# Sums across the columns (returns sum for each row)
# axis=1 is synonymous with axis='columns'
demo.sum(axis="columns")

a    1.40
b    2.60
c    0.00
d   -0.55
dtype: float64

In [857]:
# 👉 `skipna=False` says 'do NOT ignore missing values'. Any NaN in a column poisons its total.
# If any value is NaN, the result is NaN
demo.sum(skipna=False)

one   NaN
two   NaN
dtype: float64

In [858]:
# 👉 Same idea going across the rows instead of down the columns.
demo.sum(axis=1, skipna=False)

a     NaN
b    2.60
c     NaN
d   -0.55
dtype: float64

**Back to reality.** On `patients`, `axis="columns"` is almost always wrong — `age + bill_sgd`
answers no question anyone has. You will still meet it constantly, usually in two legitimate places:

- columns that genuinely add up (Q1 + Q2 + Q3 + Q4 sales, votes per candidate)
- the "does *any* column in this row have a problem?" test — `.any(axis="columns")`, which you use
  in section 2.3 to find rows containing an outlier

**One thing to remember:** `axis="index"` (or `0`) means *go down*; `axis="columns"` (or `1`) means
*go across*. The name says which direction you collapse, not which one you keep.

### 🛠️ Group Exercise 1 — Summarising (8 min)

The scaffold fades: task (a) is done for you, (b) is half-written, (c) and (d) are yours.

> **(a) Worked for you** — the cell below sorts a count table by its label instead of by count.
> Read it, then run it.
>
> **(b) Fill in the blanks** — sort ward counts by frequency, *smallest first*:
> ```python
> patients["ward"].value_counts().sort_______(ascending=____)
> ```
> *Expected:* 9 rows, smallest count at the top.
>
> **(c) From scratch** — which patient has the largest bill? Use `.idxmax()` to get the row
> label, then `.loc[...]` to pull that row out.
> *Expected:* one row, `bill_sgd` = 23100.0.
>
> **(d) Explain, no code** — `patients.info()` reports 35 non-null values for `days_admitted`
> but the table has 38 rows. Where did the other 3 go, and why is that better than showing 0?

**(a) Worked example** — count, then sort by label:

In [ ]:
# 👉 `.sort_index()` re-sorts that count table by the label (a, b, c, d) instead of by count.
#    Reading right to left: count the values, then sort the result.
obj.value_counts().sort_index()

---

# ☕ Break — 10 minutes

**Where we are:** you can now describe *what* a dataset looks like.
**Next up:** Part 2 — fixing what is wrong with it (missing values, duplicates, outliers).

---

## Part 2: Data Quality — Missing Data, Duplicates & Outliers

**Learning outcome 2:** *Handle missing values, duplicates, and outliers using appropriate Pandas methods.*

**Goal:** Part 1 found five problems in `patients`. Now we fix them, building up a `clean`
table as we go. Each fix follows the same shape: **find it → decide → apply → verify**.

⏱️ 42 min (30 min taught + 12 min group exercise)

### 2.1: Handling Missing Data

Missing data is often represented as `NaN` (Not a Number) or `None`.

**Step 1 — find the holes on our own dataset.**

In [859]:
# 👉 `.isna()` marks every cell True/False for "is this missing?". Chaining `.sum()`
#    counts the Trues per column, because Python counts True as 1 and False as 0.
patients.isna().sum()

patient_id       0
name             0
contact          3
admit_date       0
ward             0
age              0
blood_type       5
days_admitted    3
bill_sgd         0
dtype: int64

**Step 2 — decide, column by column.** There is no single right answer, only defensible ones:

| Column | Holes | Decision | Why |
|---|---|---|---|
| `contact` | 3 | **leave as NaN** | We cannot invent an email. Missing is the truth. |
| `blood_type` | 5 | **fill with `"Unknown"`** | Clinically it is a real category, not zero. |
| `days_admitted` | 3 | **fill with the median** | Numeric, and we need it for later averages. |

**Never fill a hole just because it is a hole. Ask what the missing value *means*.**

In [861]:
# 👉 `.copy()` makes an independent table so `patients` stays as the untouched original --
#    you always want the raw data available to go back to.
clean = patients.copy()

# 👉 Fill one named column. `clean["blood_type"] = ...` writes the result back into the table;
#    without that assignment, fillna would just show you a preview and change nothing.
clean["blood_type"] = clean["blood_type"].fillna("Unknown")

# 👉 Verify immediately. blood_type should now read 0.
clean.isna().sum()

patient_id       0
name             0
contact          3
admit_date       0
ward             0
age              0
blood_type       0
days_admitted    3
bill_sgd         0
dtype: int64

`days_admitted` we leave for now: it still contains a `-999` sentinel that would poison the
median. We remove impossible values first, in section 2.3, then fill. **Order matters.**

The rest of 2.1 is drills on tiny tables, so you can see each method in isolation.

In [862]:
# 👉 A Series of numbers where one entry is missing.
float_data = pd.Series([1.2, -3.5, np.nan, 0])

float_data

0    1.2
1   -3.5
2    NaN
3    0.0
dtype: float64

In [863]:
# 👉 `.isna()` answers 'is this one missing?' for every entry: True means missing.
# .isna() returns a boolean mask (True if missing, False if present)
float_data.isna()

0    False
1    False
2     True
3    False
dtype: bool

The built-in Python `None` value is also treated as NA in pandas object arrays.

In [864]:
# 👉 Python's own `None` also counts as missing, alongside `np.nan`.
string_data = pd.Series(["aardvark", np.nan, None, "avocado"])

string_data

0    aardvark
1         NaN
2        None
3     avocado
dtype: object

In [865]:
# 👉 Proof: both `np.nan` and `None` come back as True.
string_data.isna()

0    False
1     True
2     True
3    False
dtype: bool

### Strategy 1: Dropping Missing Data (`dropna`)
The simplest strategy is to just remove the rows or columns that contain missing values.

In [866]:
# 👉 `.dropna()` returns a copy with the missing entries removed. The original is unchanged --
#    pandas methods hand you a new object unless you reassign it.
s_demo = pd.Series([1, np.nan, 3.5, np.nan, 7])

# Removes all NaN values from the Series
s_demo.dropna()

0    1.0
2    3.5
4    7.0
dtype: float64

In [867]:
# 👉 The manual version of the same thing: `notna()` marks the good rows, and putting that
#    True/False list inside s_demo[...] keeps only the True ones. This is 'boolean filtering'.
# This is equivalent to boolean filtering:
s_demo[s_demo.notna()]

0    1.0
2    3.5
4    7.0
dtype: float64

With DataFrames, `dropna` by default drops **any row** containing **any missing value**.

In [868]:
# 👉 A 4-row table with missing values scattered around. No column names given, so pandas
#    numbers the columns 0, 1, 2.
demo_df = pd.DataFrame([[1., 6.5, 3.], [1., np.nan, np.nan], 
                     [np.nan, np.nan, np.nan], [np.nan, 6.5, 3.]])

demo_df

,0,1,2
0,1.0,6.5,3.0
1,1.0,NaN,NaN
2,NaN,NaN,NaN
3,NaN,6.5,3.0


In [869]:
# 👉 Default behaviour: drop a row if it has *any* missing value. Harsh -- only row 0 survives.
# Drops rows 1, 2, and 3 because they have at least one NaN
demo_df.dropna()

,0,1,2
0,1.0,6.5,3.0


We can control this behavior. `how='all'` only drops rows where **all** values are NaN.

In [870]:
# 👉 `how="all"` is gentler: only drop a row where *every* value is missing.
demo_df.dropna(how="all")

,0,1,2
0,1.0,6.5,3.0
1,1.0,NaN,NaN
3,NaN,6.5,3.0


To drop **columns** instead of rows, pass `axis=1`.

In [871]:
# 👉 Add a brand-new column called 4 that is entirely missing, so we have something to drop.
# Let's add a column of all NaNs first
demo_df[4] = np.nan

demo_df

,0,1,2,4
0,1.0,6.5,3.0,NaN
1,1.0,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN
3,NaN,6.5,3.0,NaN


In [872]:
# 👉 `axis="columns"` switches the target from rows to columns, so the all-missing column goes.
# Drops the column '4' because it is all NaNs
demo_df.dropna(axis="columns", how="all")

,0,1,2
0,1.0,6.5,3.0
1,1.0,NaN,NaN
2,NaN,NaN,NaN
3,NaN,6.5,3.0


We can also set a **threshold**: keep only rows containing at least `n` valid observations.

In [882]:
# 👉 Make a 7-row, 3-column table of random numbers, then poke holes in it.
#    `.iloc[rows, cols]` selects by position; `:4` means 'the first 4 rows'.
holes = pd.DataFrame(np.random.standard_normal((7, 3)))
# Set some missing values
holes.iloc[:4, 1] = np.nan
holes.iloc[:2, 2] = np.nan

holes

,0,1,2
0,-0.652946,NaN,NaN
1,-1.274636,NaN,NaN
2,-0.774799,NaN,2.870116
3,-0.588998,NaN,-0.643203
4,0.935602,-1.077011,0.935084
5,-0.256411,0.525049,-0.448838
6,-1.056114,0.536386,0.231892


In [883]:
# 👉 With the default settings, most rows are gone.
holes.dropna()

,0,1,2
4,0.935602,-1.077011,0.935084
5,-0.256411,0.525049,-0.448838
6,-1.056114,0.536386,0.231892


In [884]:
# 👉 `thresh=2` means 'keep the row if it has at least 2 real (non-missing) values'.
# Keep rows that have at least 2 non-NaN values
holes.dropna(thresh=2)

,0,1,2
2,-0.774799,NaN,2.870116
3,-0.588998,NaN,-0.643203
4,0.935602,-1.077011,0.935084
5,-0.256411,0.525049,-0.448838
6,-1.056114,0.536386,0.231892


### Strategy 2: Filling Missing Data (`fillna`)
Instead of losing data, we can fill the holes with a constant or a calculated value.

In [885]:
# 👉 `.fillna()` plugs every hole with a value instead of deleting the row.
# Replace all NaNs with 0
holes.fillna(0)

,0,1,2
0,-0.652946,0.000000,0.000000
1,-1.274636,0.000000,0.000000
2,-0.774799,0.000000,2.870116
3,-0.588998,0.000000,-0.643203
4,0.935602,-1.077011,0.935084
5,-0.256411,0.525049,-0.448838
6,-1.056114,0.536386,0.231892


You can specify different fill values for each column:

In [886]:
# 👉 Pass a dictionary to use a different filler per column: {column_name: fill_value}.
# Fill column 1 with 0.5, and column 2 with 0
holes.fillna({1: 0.5, 2: 0})

,0,1,2
0,-0.652946,0.500000,0.000000
1,-1.274636,0.500000,0.000000
2,-0.774799,0.500000,2.870116
3,-0.588998,0.500000,-0.643203
4,0.935602,-1.077011,0.935084
5,-0.256411,0.525049,-0.448838
6,-1.056114,0.536386,0.231892


**Forward/Backward Fill:** Useful for time-series data, where you propagate the last valid observation forward or backward.

In [892]:
# 👉 `bfill` = backward fill: copy the next real value upwards into the gap.
#    Common with time-series data where the neighbouring reading is a fair guess.
# Propagate next valid value backward to fill gaps
holes.bfill()

,0,1,2
0,-0.652946,-1.077011,2.870116
1,-1.274636,-1.077011,2.870116
2,-0.774799,-1.077011,2.870116
3,-0.588998,-1.077011,-0.643203
4,0.935602,-1.077011,0.935084
5,-0.256411,0.525049,-0.448838
6,-1.056114,0.536386,0.231892


In [891]:
# 👉 `limit=2` stops the copying after 2 rows, so long gaps are not silently invented.
# Same, but limit how many rows are filled consecutively
holes.bfill(limit=2)

,0,1,2
0,-0.652946,NaN,2.870116
1,-1.274636,NaN,2.870116
2,-0.774799,-1.077011,2.870116
3,-0.588998,-1.077011,-0.643203
4,0.935602,-1.077011,0.935084
5,-0.256411,0.525049,-0.448838
6,-1.056114,0.536386,0.231892


**Imputation:** Filling with the mean or median is a very common technique.

In [893]:
# 👉 A small Series with two holes in it.
s_holes = pd.Series([1., np.nan, 3.5, np.nan, 7])

s_holes

0    1.0
1    NaN
2    3.5
3    NaN
4    7.0
dtype: float64

In [897]:
# 👉 'Imputation': fill the holes with the average of the values we do have.
#    `s_holes.mean()` is calculated first, then handed to `fillna`.
# Fill with the mean of the available s_holes
print(s_holes.mean())
s_holes.fillna(s_holes.mean())

3.8333333333333335


0    1.000000
1    3.833333
2    3.500000
3    3.833333
4    7.000000
dtype: float64

### 2.2: Handling Duplicates

Duplicate rows can skew analysis and models. We typically identify them and remove them.

**On our dataset:** the ritual said 38 rows but only 36 unique patient IDs.

In [898]:
# 👉 `.duplicated()` flags a row True if an identical row appeared earlier.
#    `.sum()` counts those flags.
clean.duplicated().sum()

2

In [899]:
# 👉 Show the offending rows so you can eyeball them before deleting anything.
#    `keep=False` flags *every* copy, not just the repeats, so you see both rows of each pair.
clean[clean.duplicated(keep=False)].sort_values("patient_id")

,patient_id,name,contact,admit_date,ward,age,blood_type,days_admitted,bill_sgd
13,P014,Nurul Huda,nurul.huda@example.com,2024-01-13,Cardiology,71,O-,8.0,6980.0
37,P014,Nurul Huda,nurul.huda@example.com,2024-01-13,Cardiology,71,O-,8.0,6980.0
20,P021,Umar Farouk,umar.farouk@example.com,2024-01-20,General,31,Unknown,2.0,1340.0
36,P021,Umar Farouk,umar.farouk@example.com,2024-01-20,General,31,Unknown,2.0,1340.0


In [900]:
# 👉 Genuine full-row copies, so drop them. Assign back for the change to stick.
clean = clean.drop_duplicates()

clean.shape

(36, 9)

36 rows. **Note the safer habit:** here the whole row was identical, so `drop_duplicates()`
is safe. If two rows shared a `patient_id` but differed elsewhere, that is a *conflict*, not a
duplicate — you would investigate, not delete. `subset=["patient_id"]` (below) would silently
pick one and bin the other.

Drills on a tiny table follow.

In [901]:
# 👉 Build a table from a dictionary: each key becomes a column name, each list becomes the
#    column's values. `["one", "two"] * 3` repeats the list 3 times -- a handy Python shortcut.
dupes = pd.DataFrame({"k1": ["one", "two"] * 3 + ["two"],
                     "k2": [1, 1, 2, 3, 3, 4, 4]})

dupes

,k1,k2
0,one,1
1,two,1
2,one,2
3,two,3
4,one,3
5,two,4
6,two,4


`duplicated()` returns a boolean Series indicating whether each row has been seen before.

In [902]:
# 👉 For each row: 'have I already seen this exact row above?' True means it is a repeat.
dupes.duplicated()

0    False
1    False
2    False
3    False
4    False
5    False
6     True
dtype: bool

`drop_duplicates()` creates a new DataFrame with the duplicates removed.

In [903]:
# 👉 Keep only the first appearance of each row and throw the repeats away.
dupes.drop_duplicates()

,k1,k2
0,one,1
1,two,1
2,one,2
3,two,3
4,one,3
5,two,4


**Subset:** Sometimes we only care about duplicates in specific columns.

In [904]:
# 👉 Add a column of running numbers so we can see which rows survive the next steps.
dupes["v1"] = range(7)

dupes

,k1,k2,v1
0,one,1,0
1,two,1,1
2,one,2,2
3,two,3,3
4,one,3,4
5,two,4,5
6,two,4,6


In [905]:
# 👉 `subset=` narrows the comparison: rows count as duplicates if column k1 matches,
#    even when the other columns differ.
# Drop duplicates considering only column 'k1'
dupes.drop_duplicates(subset=["k1"])

,k1,k2,v1
0,one,1,0
1,two,1,1


**Keep:** By default, it keeps the first occurrence. We can keep the last one instead.

In [906]:
# 👉 By default pandas keeps the *first* of each duplicate group. `keep="last"` keeps the last.
dupes.drop_duplicates(subset=["k1", "k2"], keep="last")

,k1,k2,v1
0,one,1,0
1,two,1,1
2,one,2,2
3,two,3,3
4,one,3,4
6,two,4,6


### 2.3: Handling Outliers

Outliers are extreme values that deviate significantly from the rest of the data. We can filter them using boolean indexing.

**Wait — why do we care about outliers?**

A patient's age reads 150 — is that a typo to fix, or a real value to keep? Data that's technically valid can still be impossible. Let's use the IQR rule to flag suspicious values before they poison our averages.

**On our dataset:** three impossible-value problems, and they need different treatments.

In [907]:
# 👉 Boolean filtering: build a True/False test, put it in the square brackets, and only
#    the True rows come back. `|` means OR (and each side needs its own brackets).
clean[(clean["age"] > 100) | (clean["age"] < 0)]

,patient_id,name,contact,admit_date,ward,age,blood_type,days_admitted,bill_sgd
3,P004,Divya Nair,divya.nair@example.org,2024-01-05,icu,150,O-,9.0,15200.0
9,P010,Jasmine Koh,jasmine.koh@example.com,2024-01-10,Cardiology,-3,A+,6.0,4410.0


Two rows. Now the judgement call this section is really about:

- Age **150** is almost certainly a typo (50? 15?). We cannot know — so mark it missing rather
  than invent a number.
- Age **-3** is impossible, same treatment.
- `days_admitted` **-999** and `bill_sgd` **-999** are not outliers at all: they are a *sentinel*,
  someone's code for "no value recorded". Marking them missing is a correction, not a judgement.

**Capping would be wrong here.** Capping age at 100 would state that this patient is 100 years
old — inventing data. Cap when the extreme value is real but you want to limit its influence;
mark missing when the value is simply not credible.

In [908]:
# 👉 `.mask(condition)` replaces values where the condition is True with NaN.
#    Read it as "hide the values that fail my sanity check".
clean["age"] = clean["age"].mask((clean["age"] > 100) | (clean["age"] < 0))

# 👉 Sentinels: -999 never means a real quantity in this dataset.
clean["days_admitted"] = clean["days_admitted"].mask(clean["days_admitted"] == -999)
clean["bill_sgd"] = clean["bill_sgd"].mask(clean["bill_sgd"] == -999)

clean[["age", "days_admitted", "bill_sgd"]].describe()

,age,days_admitted,bill_sgd
count,34.000000,32.000000,35.000000
mean,51.264706,6.312500,7745.890000
std,14.523354,4.387942,7446.995518
min,26.000000,1.000000,760.000000
25%,39.500000,3.000000,2145.625000
50%,51.000000,5.000000,4410.000000
75%,64.500000,9.250000,15840.000000
max,74.000000,16.000000,23100.000000


Now the ranges are plausible — and we have created new holes on purpose. Fill them, in the
right order: sentinels out first, *then* compute the median.

In [910]:
# 👉 Median, not mean: the median ignores lopsided extremes, so it is the safer default
#    for imputing. We compute it from the sane values that remain.
clean["age"] = clean["age"].fillna(clean["age"].median())
clean["days_admitted"] = clean["days_admitted"].fillna(clean["days_admitted"].median())
clean["bill_sgd"] = clean["bill_sgd"].fillna(clean["bill_sgd"].median())

# 👉 Verify: only `contact` should still have holes, and that was deliberate.
clean.isna().sum()

patient_id       0
name             0
contact          3
admit_date       0
ward             0
age              0
blood_type       0
days_admitted    0
bill_sgd         0
dtype: int64

**Compare with the original to see what cleaning bought us:**

In [911]:
# 👉 Same statistic, dirty vs clean. This is the number that would have gone into a report.
print("mean age  — raw:", round(patients["age"].mean(), 1),
      "| clean:", round(clean["age"].mean(), 1))
print("max age   — raw:", patients["age"].max(),
      "| clean:", clean["age"].max())
print("mean stay — raw:", round(patients["days_admitted"].mean(), 1),
      "| clean:", round(clean["days_admitted"].mean(), 1))

mean age  — raw: 52.4 | clean: 51.2
max age   — raw: 150 | clean: 74.0
mean stay — raw: -22.5 | clean: 6.2


The raw mean stay is *negative* — a single `-999` was enough. Drills on random data follow,
where you can see the standard-deviation and IQR rules on a clean bell curve.

In [912]:
# 👉 1000 rows of random numbers shaped like a bell curve, so most values sit near 0.
#    Values far from 0 are our stand-in for outliers.
# Create a dataset with normal distribution
spread = pd.DataFrame(np.random.standard_normal((1000, 4)))

spread.describe()

,0,1,2,3
count,1000.000000,1000.000000,1000.000000,1000.000000
mean,0.073667,-0.009380,-0.004581,0.000738
std,0.995139,1.012474,0.988520,0.978131
min,-3.484708,-3.671067,-3.164824,-3.531329
25%,-0.576110,-0.716105,-0.693569,-0.666096
50%,0.058606,-0.040032,0.009347,-0.034340
75%,0.728231,0.653408,0.666841,0.666127
max,3.052611,3.108665,3.378134,3.853392


**Detection:** Let's find values exceeding 3 in absolute value (Standard Deviation > 3).

In [913]:
# 👉 Grab one column, then keep only the entries more than 3 away from zero.
#    `.abs()` ignores the minus sign, so this catches both very high and very low values.
col = spread[2]

# Boolean indexing to find rows where absolute value > 3
col[col.abs() > 3]

55     3.153375
102   -3.028216
258    3.378134
777   -3.164824
Name: 2, dtype: float64

To find **any row** that has an outlier in **any column**, we use `.any(axis=1)`.

In [914]:
# 👉 Same test applied to the whole table. `.any(axis="columns")` asks, per row,
#    'was there at least one extreme value anywhere in this row?'
# 1. spread.abs() > 3 returns a boolean DataFrame
# 2. .any(axis="columns") checks if any value in the row is True
spread[(spread.abs() > 3).any(axis="columns")]

,0,1,2,3
55,-1.481230,-0.860704,3.153375,0.892789
89,0.280773,-0.167376,1.196906,3.853392
98,-0.670121,-0.009421,0.970287,-3.531329
102,0.411105,-0.025831,-3.028216,1.323445
109,-3.484708,1.162167,0.166648,0.150850
114,0.013367,-3.671067,-0.680716,0.594180
258,1.124487,1.098553,3.378134,2.126706
281,-0.648116,0.069542,-0.661886,-3.141375
314,-3.110265,1.942562,0.447152,0.191126
531,3.052611,0.557463,0.792351,0.908866


**Capping:** Instead of removing outliers, we can cap them at a threshold.

In [915]:
# 👉 'Capping': pull extreme values back to the ±3 boundary instead of deleting the row.
#    `np.sign(spread)` is +1 or -1, so multiplying by 3 keeps the original direction.
# Set values > 3 to 3, and < -3 to -3, preserving the sign
spread[spread.abs() > 3] = np.sign(spread) * 3

In [916]:
# 👉 Re-run the summary and compare: min and max are now exactly -3 and 3.
spread.describe()

,0,1,2,3
count,1000.000000,1000.000000,1000.000000,1000.000000
mean,0.074210,-0.008508,-0.004919,0.000422
std,0.993016,1.008968,0.986216,0.972480
min,-3.000000,-3.000000,-3.000000,-3.000000
25%,-0.576110,-0.716105,-0.693569,-0.666096
50%,0.058606,-0.040032,0.009347,-0.034340
75%,0.728231,0.653408,0.666841,0.666127
max,3.000000,3.000000,3.000000,3.000000


**Removal:** Or we can just drop the rows with outliers.

In [917]:
# 👉 The other option -- 'trimming': keep only rows where *every* column is within bounds.
# Keep rows where ALL columns are within the threshold ( < 3)
spread[(spread.abs() < 3).all(axis="columns")]

,0,1,2,3
0,0.362639,-0.067469,-0.769051,0.202645
1,0.946831,-1.579360,0.632176,-0.984366
2,0.790627,-1.119383,-1.710894,-0.100453
3,0.509094,-0.272796,0.752805,-0.494644
4,-0.587550,-1.052824,0.009130,0.671609
...,...,...,...,...
995,0.587133,-1.036275,0.645498,0.353058
996,1.679537,1.406949,0.488660,-0.284669
997,0.247831,-0.988951,-1.510834,-1.238350
998,-0.190109,0.227470,-0.433295,0.953133


### 🛠️ Group Exercise 2 — Data Quality (12 min)

Same fading. Work on `practice` (created below) — a small random table with holes punched in it,
so every change is visible.

> **(a) Fill in the blanks** — drop rows that are missing *everything*:
> ```python
> practice.dropna(how="____")
> ```
> *Expected:* 5 of the 6 rows survive — exactly one row is entirely empty.
>
> **(b) Half-written** — keep only rows with at least 2 real values:
> ```python
> practice.dropna(thresh=____)
> ```
> *Expected:* 4 rows survive. Work out which two are dropped, and why, before running it.
>
> **(c) From scratch** — fill every hole with that column's median. One line.
> *Expected:* `practice.isna().sum()` is 0 for all three columns afterwards.
>
> **(d) From scratch, duplicates** — append a copy of the first row with
> `pd.concat([practice, practice.iloc[[0]]])`, then detect and remove it.
> *Expected:* `duplicated().sum()` goes 1 → 0.
>
> **(e) Judgement, no code** — a patient's `bill_sgd` reads 250000, thirty times the next
> highest. Cap it, mark it missing, or leave it? Defend your answer in one sentence, then say
> what extra information would change your mind.

**(a)–(d)** use the `practice` table below.

In [918]:
# 👉 Practice data for the exercise: 6 rows of random numbers with holes punched in three
#    places. `practice.iloc[4:, 2]` means 'row 4 onward, column 2'.
practice = pd.DataFrame(np.random.standard_normal((6, 3)))

practice.iloc[[2,4,5], 1] = np.nan
practice.iloc[4:, 2] = np.nan
practice.iloc[3:5, 0] = np.nan

practice

,0,1,2
0,1.447334,-0.803936,-0.727066
1,-1.403307,-0.276108,0.159528
2,-0.672365,NaN,-0.411090
3,NaN,0.171355,-1.025050
4,NaN,NaN,NaN
5,-0.519038,NaN,NaN


> **Stuck?** Every method you need appeared in 2.1–2.3 above. The pattern is always:
> build the test, look at what it catches, then apply it and verify.

---

# ☕ Break — 10 minutes

**Where we are:** the data is now clean.
**Next up:** Part 3 — reshaping clean data into the form an analysis or model needs.

---

## Part 3: Data Transformation

**Learning outcome 3:** *Transform data through type conversion, string cleaning, and categorical encoding.*

**Goal:** `clean` has no holes or duplicates now, but it still is not *useful*: nine ward
spellings for four wards, dates stored as text, no way to compare wards. We fix all three.

⏱️ 35 min (25 min taught + 10 min group exercise)

### 3.1: Transforming Data (Mapping)

Sometimes we need to add new columns based on existing ones. `map()` is perfect for this.

**On our dataset:** nine ward spellings for four wards. A dictionary maps every messy
spelling to one canonical name — this is exactly what `.map()` and `.replace()` are for.

In [ ]:
# 👉 See the mess in full before writing the mapping. `.unique()` lists distinct values.
clean["ward"].unique()

In [ ]:
# 👉 A dictionary is a lookup table: {what_is_in_the_data: what_we_want}.
#    Writing it out by hand is fine and normal for a handful of categories.
ward_fix = {
    "ICU": "ICU", " ICU ": "ICU", "icu": "ICU", "I.C.U": "ICU",
    "General": "General", "GENERAL": "General",
    "Orthopaedics": "Orthopaedics", "orthopaedics": "Orthopaedics",
    "Cardiology": "Cardiology",
}

# 👉 `.map()` swaps each value for its dictionary match. Every value must be in the dictionary,
#    or you get NaN -- which is why we printed `.unique()` first.
clean["ward"] = clean["ward"].map(ward_fix)

clean["ward"].value_counts()

Four wards. **A warning worth internalising:** if a tenth spelling appears next month,
`.map()` turns it into NaN silently. `.replace()` leaves unlisted values alone instead — safer for
partial fixes, more dangerous when you *want* to catch surprises. Pick deliberately.

In 3.3 you will see the scalable version of this same fix using `.str` methods.

Drills below.

In [ ]:
# 👉 A tiny table of foods and weights, built from a dictionary of column-name: values.
foods = pd.DataFrame({"food": ["bacon", "pulled pork", "bacon", "pastrami", 
                                "corned beef", "bacon", "pastrami", "honey ham", 
                                "nova lox"],
                         "ounces": [4, 3, 12, 6, 7.5, 8, 3, 5, 6]})

foods

**Scenario:** We want to add a column indicating the animal source of each food.

In [ ]:
# 👉 A dictionary = a lookup table. Given a food (the key) it hands back an animal (the value).
meat_to_animal = {
    "bacon": "pig",
    "pulled pork": "pig",
    "pastrami": "cow",
    "corned beef": "cow",
    "honey ham": "pig",
    "nova lox": "salmon"
}

In [ ]:
# 👉 `.map()` walks down the food column and swaps each value for its dictionary match.
#    Assigning to foods["animal"] creates the new column.
# .map() looks up the value in the 'food' column in our dictionary
foods["animal"] = foods["food"].map(meat_to_animal)

foods

You can also pass a function to `map()` for custom logic.

In [ ]:
# 👉 `.map()` also accepts a function. `def` defines one: it takes an input x and returns
#    the matching animal. Same result as passing the dictionary directly.
def get_animal(x):
    return meat_to_animal[x]

foods["food"].map(get_animal)

**Replacing Values:** `replace` is a specialized version of map, great for fixing sentinel values (like -999 for missing data).

In [ ]:
# 👉 Some datasets use a fake number like -999 to mean 'no sentinel_s recorded'.
sentinel_s = pd.Series([1, -999, 2, -999, -1000, 3])

sentinel_s

In [ ]:
# 👉 `.replace()` swaps one value for another -- here, the fake code becomes a proper NaN.
# Replace -999 with NaN
sentinel_s.replace(-999, np.nan)

In [ ]:
# 👉 Pass a list to replace several values with the same thing in one go.
# Replace multiple values at once
sentinel_s.replace([-999, -1000], np.nan)

In [ ]:
# 👉 Two lists of equal length: first list is what to find, second is what to put in its place.
# Replace with different values (-999 -> NaN, -1000 -> 0)
sentinel_s.replace([-999, -1000], [np.nan, 0])

In [ ]:
# 👉 The clearest form: a dictionary of {old_value: new_value}. Same result, easier to read.
# Using a dictionary for clarity
sentinel_s.replace({-999: np.nan, -1000: 0})

### 3.2: Renaming Axis Indexes
Changing row/column labels using mapping.

In [ ]:
# 👉 A 3x4 table of the numbers 0-11. `np.arange(12)` makes 0..11 in a line and
#    `.reshape((3, 4))` folds that line into 3 rows of 4.
labels = pd.DataFrame(np.arange(12).reshape((3, 4)),
                    index=['Ohio', 'Colorado', 'New York'],
                    columns=['one', 'two', 'three', 'four'])

In [ ]:
# 👉 Display it. Note the row labels are place names, not numbers.
labels

Using `.map()` on the index:

In [ ]:
# 👉 A function that shortens a label to its first 4 characters and upper-cases it.
#    `x[:4]` is Python slicing: 'characters from the start up to position 4'.
def transform(x):
    return x[:4].upper()

labels.index.map(transform)

In [ ]:
# 👉 `.index.map()` applies that function to every row label. Assigning back to `labels.index`
#    is what actually changes the table -- the previous cell only previewed the result.
# Assign back to modify in-place
labels.index = labels.index.map(transform)

In [ ]:
# 👉 Confirm the row labels changed.
labels

Using `.rename()` (returns a copy by default):

In [ ]:
# 👉 `.rename()` is the tidier way to relabel. `str.title` and `str.upper` are ready-made
#    functions, so no `def` needed. This returns a copy; the original is untouched.
labels.rename(index=str.title, columns=str.upper)

In [ ]:
# 👉 `.rename()` also takes dictionaries when you only want to change specific labels.
labels.rename(index={"OHIO": "INDIANA"}, columns={"three": 3})

### 3.3: String Manipulation

Pandas has a special accessor `.str` that unlocks string methods for an entire Series at once. This handles missing values gracefully.

**On our dataset:** the `name` column has stray spaces and inconsistent case, and we want
the email provider out of `contact`.

In [ ]:
# 👉 Find the problem first. `repr()` shows the invisible characters, so spaces become visible.
[repr(n) for n in clean["name"].head(8)]

In [ ]:
# 👉 `.str` applies a text method to the whole column at once. Chain them left to right:
#    strip the outer spaces, then Title Case The Words.
clean["name"] = clean["name"].str.strip().str.title()

[repr(n) for n in clean["name"].head(8)]

That two-method chain would have fixed the ward column too (`.str.strip().str.upper()`)
without writing a nine-entry dictionary. Both are valid; `.str` scales, the dictionary documents.

In [ ]:
# 👉 `.str.split("@")` cuts each email in two at the @ sign; `.str[1]` takes the second
#    piece. Python counts from 0, so [1] is the second item. Missing emails stay missing.
clean["email_domain"] = clean["contact"].str.split("@").str[1]

clean[["contact", "email_domain"]].head()

In [ ]:
# 👉 Start from a dictionary of name: email, then turn it into a Series so the names become
#    row labels. One email is missing on purpose.
emails = {"Dave": "dave@google.com", "Steve": "steve@gmail.com",
        "Rob": "rob@gmail.com", "Wes": np.nan}

emails = pd.Series(emails)

emails

In [ ]:
# 👉 `.str` unlocks text methods for a whole column at once. Note the missing entry stays NaN --
#    pandas will not guess an answer for emails it does not have.
# Check if 'gmail' exists in each string
emails.str.contains("gmail")

Note on Data Types: Pandas has a specialized `StringDType` (`string`) vs the generic `object` type.

In [ ]:
# 👉 Convert to the dedicated text type. Mostly the same, but missing values behave more
#    predictably (you get <NA> instead of NaN).
emails_str = emails.astype('string')

emails_str

In [ ]:
# 👉 Same test on the text-typed column: now the answer is a proper True/False/<NA>.
emails_str.str.contains("gmail")

**Slicing:** We can treat the column like a Python string.

In [ ]:
# 👉 `.str[:5]` slices every value to its first 5 characters.
# Get the first 5 characters
emails_str.str[:5]

**Regex — one step at a time.** A *regular expression* is a pattern that describes the
shape of text. It looks alarming; it is just three ideas stacked. We build up to the full pattern
in three steps, so you never meet more than one new idea at a time.

In [ ]:
# 👉 `re` is Python's built-in regular-expression module -- pattern matching for text.
import re

**Step 1 — a pattern with no groups.** `.` means "any one character" and `+` means
"one or more of the thing before me". So `@.+` reads: an @ sign followed by at least one character.

In [ ]:
# 👉 `.str.contains` takes a regex, not just plain text. This asks: is there an @ sign
#    with something after it? Every real email matches; the missing one stays NaN.
emails.str.contains(r"@.+")

**Step 2 — one group.** Round brackets `( )` mark the part you want to *keep*.
`.str.extract` returns whatever is inside them, and throws the rest away.

In [ ]:
# 👉 Read the pattern as: "an @ sign, then capture everything after it".
#    That capture is the email's domain plus suffix.
emails.str.extract(r"@(.+)")

**Step 3 — three groups.** Same idea, three times, plus two extra pieces of notation:

| Piece | Means |
|---|---|
| `[A-Z0-9._%+-]` | any one character from this set |
| `+` | one or more of the previous thing |
| `( )` | capture this part |
| `\.` | a literal dot (a bare `.` would mean "any character") |
| `{2,4}` | between 2 and 4 of the previous thing |

`r"..."` tells Python to leave backslashes alone, and `flags=re.IGNORECASE` lets the uppercase
`A-Z` in the pattern match lowercase letters too.

In [ ]:
# 👉 A regex pattern with three bracketed groups: username, domain, suffix.
#    The `r"..."` prefix means 'treat backslashes literally'. `flags=re.IGNORECASE` lets
#    the uppercase A-Z in the pattern match lowercase letters too.
# Pattern to identify email parts: (username) @ (domain) . (suffix)
pattern = r"([A-Z0-9._%+-]+)@([A-Z0-9.-]+)\.([A-Z]{2,4})"

emails.str.findall(pattern, flags=re.IGNORECASE)

**Retrieving elements:** We can chain `.str` calls to get specific parts of the regex match.

In [ ]:
# 👉 `.findall` returns a list of matches per row; `.str[0]` takes the first (and only) one,
#    leaving one tuple of three pieces per email.
matches = emails.str.findall(pattern, flags=re.IGNORECASE).str[0]

matches

In [ ]:
# 👉 Pull item 1 out of each tuple. Python counts from 0, so 1 is the middle piece: the domain.
# Get index 1 of the tuple (the domain name)
matches.str.get(1)

The `extract` method is very powerful—it creates a new DataFrame with columns for each captured regex group.

In [ ]:
# 👉 `.extract()` is the friendlier tool: it turns each regex group straight into its own column.
emails.str.extract(pattern, flags=re.IGNORECASE)

### 3.4: Categorical Data
Converting string columns to `category` dtype saves memory and speeds up operations.

**On our dataset:** age is a continuous number, but the report needs age *bands*.

In [ ]:
# 👉 `pd.cut` turns numbers into labelled bands. `bins` are the cut points and `labels`
#    names them -- there must be exactly one label per band (5 edges make 4 bands).
age_bins = [0, 30, 50, 70, 120]
age_labels = ["Under 30", "30-49", "50-69", "70+"]

clean["age_group"] = pd.cut(clean["age"], bins=age_bins, labels=age_labels)

clean["age_group"].value_counts()

In [ ]:
# 👉 One-hot encoding: turn one text column into several 0/1 columns, one per ward.
#    This is the form most machine-learning models require. `prefix=` labels the new columns.
pd.get_dummies(clean["ward"], prefix="ward").head()

Drills on smaller examples follow.

In [ ]:
# 👉 A column of repeated fruit names. `* 2` repeats the list, giving 8 rows.
fruit_s = pd.Series(["apple", "orange", "apple", "apple"] * 2)

fruit_s

In [ ]:
# 👉 Only two distinct fruit_s exist, even though there are 8 rows.
fruit_s.unique()

In [ ]:
# 👉 How many of each. This repetition is exactly what the `category` type optimises.
fruit_s.value_counts()

**Using Pandas `category` type:**

In [ ]:
# 👉 Build a realistic little dataset. `rng` is a random-number generator with a fixed seed,
#    so everyone in the room gets the same numbers. `columns=` fixes the column order.
fruits = ["apple", "orange", "apple", "apple"] * 2
N = len(fruits)

# to ensure reproducibility
rng = np.random.default_rng(seed=12345)

baskets = pd.DataFrame({"fruit": fruits, 
                   "basket_id": np.arange(N), 
                   "count": rng.integers(3, 15, size=N), 
                   "weight": rng.uniform(0, 4, size=N)}, 
                  columns=["basket_id", "fruit", "count", "weight"])

baskets

In [ ]:
# 👉 `.astype('category')` asks pandas to do the codes-plus-lookup-table trick for us.
#    Notice the output now reports its Categories at the bottom.
# Convert object column to category
fruit_cat = baskets['fruit'].astype('category')

fruit_cat

In [ ]:
# 👉 Save the converted column back into the table so the change actually sticks.
# Assign back to the dataframe
baskets['fruit'] = baskets['fruit'].astype('category')

baskets

**Binning Data (`pd.cut`):** Converting continuous data (e.g., Age) into categorical bins (e.g., Age Groups).

In [ ]:
# 👉 A plain Python list of ages -- continuous numbers we are about to group.
ages = [20, 22, 25, 27, 21, 23, 37, 31, 61, 45, 41, 32]

In [ ]:
# 👉 'Binning': turn numbers into labelled ranges. `bins` are the cut points, so we get
#    the groups 18-25, 25-35, 35-60, 60-100. A round bracket means the edge is excluded,
#    a square bracket means it is included: (18, 25] is 'over 18, up to and including 25'.
# Define bin edges
bins = [18, 25, 35, 60, 100]

# Cut the data
age_cat = pd.cut(ages, bins)

age_cat

In [ ]:
# 👉 Which bin each age landed in, as a code number.
age_cat.codes

In [ ]:
# 👉 The list of bin ranges that were created.
age_cat.categories

In [ ]:
# 👉 Just the first bin. Square brackets index into a list; Python counts from 0.
age_cat.categories[0]

In [ ]:
# 👉 Count how many ages fell into each bin -- an instant histogram in table form.
age_cat.value_counts()

**Binning Parameters:**

In [ ]:
# 👉 `right=False` flips which edge is included: now 18 counts, 25 starts the next bin.
# right=False makes the left side closed [inclusive, exclusive)
pd.cut(ages, bins, right=False)

In [ ]:
# 👉 Give the bins human-readable names instead of number ranges. The list of labels must
#    have exactly one entry per bin.
# Adding custom labels
group_names = ["Youth", "YoungAdult", "MiddleAged", "Senior"]

pd.cut(ages, bins, labels=group_names)

If you pass an integer number of bins to `pd.cut`, it will compute equal-length bins based on the min/max of the data.

In [ ]:
# 👉 Pass a plain number instead of edges and pandas splits the range into that many
#    equal-width bins for you. `precision=2` just shortens the decimals in the labels.
uniform_vals = np.random.uniform(size=20)

pd.cut(uniform_vals, 4, precision=2)

**Computing Indicator / Dummy Variables:** Converting categorical variables into binary columns (One-Hot Encoding) for machine learning.

In [ ]:
# 👉 A small table with a text 'key' column, ready for encoding.
keys_df = pd.DataFrame({"key": ["b", "b", "a", "c", "a", "b"], 
                   "data1": range(6)})

keys_df

In [ ]:
# 👉 'One-hot encoding': turn one text column into several 0/1 columns, one per value.
#    This is how most machine-learning models want categories fed to them.
pd.get_dummies(keys_df["key"])

In [ ]:
# 👉 `prefix=` puts the original column name in front, so you can tell where the
#    columns came from once several are combined.
dummies = pd.get_dummies(keys_df["key"], prefix="key")

dummies

Recipe: Combining `get_dummies` with `cut`.

In [ ]:
# 👉 `np.random.seed` fixes the randomness so your numbers match the notebook's.
np.random.seed(12345)

values = np.random.uniform(size=10)

values

In [ ]:
# 👉 Combine the two ideas: `cut` groups the numbers into bins, then `get_dummies` turns
#    each bin into its own 0/1 column. Read it inside-out.
bins = [0, 0.2, 0.4, 0.6, 0.8, 1]

pd.get_dummies(pd.cut(values, bins))

### 3.5: Type Conversion and Grouping

Two jobs left. `admit_date` is still **text**, so we cannot ask date questions of it. And we
still have no way to compare wards — which is the question anyone actually wants answered.

⏱️ ~7 min

In [ ]:
# 👉 Right now the dates are just strings. `.dtype` on one column confirms it: `object`.
clean["admit_date"].dtype

In [ ]:
# 👉 `pd.to_datetime` parses text into real dates. Only after this can pandas do date maths.
clean["admit_date"] = pd.to_datetime(clean["admit_date"])

clean["admit_date"].dtype

In [ ]:
# 👉 `.dt` is the date accessor, the way `.str` is for text. Now these questions are possible:
print("first admission:", clean["admit_date"].min().date())
print("last admission: ", clean["admit_date"].max().date())

# 👉 Pull parts out of a date to group by later.
clean["admit_month"] = clean["admit_date"].dt.month

clean[["admit_date", "admit_month"]].head()

**Grouping: split → apply → combine.** `groupby` splits the rows into groups, applies a
calculation to each group, and combines the answers into one table. It is the single most
useful summarising tool in pandas.

In [ ]:
# 👉 Split by ward, then average the bill within each ward. Read it as a sentence:
#    "group by ward, take the bill column, give me the mean".
clean.groupby("ward")["bill_sgd"].mean()

In [ ]:
# 👉 Several statistics at once with `.agg()`. Each line reads: new_column_name =
#    (which column to use, which calculation). "count" counts rows, "mean" averages them.
ward_summary = clean.groupby("ward").agg(
    patients=("patient_id", "count"),
    avg_age=("age", "mean"),
    avg_days=("days_admitted", "mean"),
    avg_bill=("bill_sgd", "mean"),
).round(1)

ward_summary

**This table is the point of the whole lesson.** It is only trustworthy because every
number behind it was cleaned first: ICU's average stay would have been negative with the `-999`
still in place, and the ward rows would have been split nine ways.

Compare — the same summary on the raw data:

In [ ]:
# 👉 The same question asked of the dirty table. Nine ward groups, and a negative average.
patients.groupby("ward")["days_admitted"].mean().round(1)

### 🛠️ Group Exercise 3 — Transformation (10 min)

Split (a), (b), (c) across your group, then compare. Scaffolding fades as you go.

> **(a) Fill in the blanks — mapping** — on `practice3` below, replace both sentinels at once:
> ```python
> practice3.replace({-999: ____, 999: ____})
> ```
> *Expected:* -999 becomes NaN, 999 becomes 0, everything else untouched.
>
> **(b) Half-written — strings** — pull the username (the part *before* the @) out of
> `clean["contact"]`:
> ```python
> clean["contact"].str.split("____").str[____]
> ```
> *Expected:* `aisha.rahman`, `brandon.lee`, ... and NaN where the email was missing.
>
> **(c) From scratch — categories and grouping** — build a table with one row per `age_group`
> showing how many patients and their average bill. Use `groupby` + `.agg`, as in 3.5.
> *Expected:* 4 rows, one per band.
>
> **(d) Stretch** — redo (c) grouped by *two* columns at once: `groupby(["ward", "age_group"])`.
> Look at the result and say in one sentence why so many combinations are empty.

**(a)** uses the `practice3` table below.

In [ ]:
# 👉 Practice data for the exercise: -999 and 999 stand in for two different bad codes.
practice3 = pd.DataFrame(np.random.standard_normal((6, 3)))

practice3.iloc[2:, 1] = -999
practice3.iloc[4:, 2] = 999

practice3

> **Check yourself:** `practice3.replace(...)` returns a *new* table. Nothing changes
> in `practice3` unless you assign the result back to it.

**(b)** uses `clean["contact"]` from section 3.3.

**(c)** and **(d)** use `clean` and the `age_group` column from section 3.4.

> **Reminder:** `.agg(new_name=(column, calculation))` — one line per statistic.

---

## Part 4: Reading and Writing Data

**Learning outcome 4:** *Read and write data across multiple file formats (CSV, JSON, Excel, databases).*

**Goal:** Everything so far used DataFrames we built by hand. This closing section is the plumbing:
getting real files in, and your cleaned results back out. Read the core `read_csv` / `to_csv` cells
live; the **OPTIONAL / REFERENCE** blocks are lookup material for when you meet that format.

⏱️ 15 min

### 4.1: Reading Data

Pandas is incredibly flexible with input formats. Let's start by reading a standard Comma-Separated Values (CSV) file.

In [ ]:
# 👉 The leading `!` runs a terminal command rather than Python. `cat` prints a file as-is,
#    so we can see the raw text before pandas touches it.
# Let's inspect the raw file content first using a shell command
!cat ../data/ex1.csv

In [ ]:
# 👉 `read_csv` turns that text file into a DataFrame. The path `../data/` means
#    'go up one folder, then into data'.
# Use read_csv to load the data into a DataFrame
# By default, it assumes the first row is the header
df = pd.read_csv("../data/ex1.csv")

df

**Scenario:** What if the file doesn't have a header row? If we don't specify this, pandas will mistakenly use the first row of data as column names.

In [ ]:
# 👉 Same peek, at a file whose first line is real data rather than column names.
!cat ../data/ex2.csv

In [ ]:
# 👉 Without a header row, pandas would steal the first data row for names.
#    `header=None` prevents that and numbers the columns instead.
# Option 1: Tell pandas there is no header. It will assign integers (0, 1, 2...) as column names.
pd.read_csv("../data/ex2.csv", header=None)

In [ ]:
# 👉 Better: supply your own column names with `names=`.
# Option 2: Provide your own column names using the 'names' parameter
pd.read_csv("../data/ex2.csv", names=["a", "b", "c", "d", "message"])

**Indexing:** You can also designate a specific column to be the index (row labels) of the DataFrame, rather than the default 0, 1, 2... index.

In [ ]:
# 👉 `index_col=` promotes one column to be the row labels instead of an ordinary column.
names = ["a", "b", "c", "d", "message"]

# Use 'index_col' to set the 'message' column as the index
pd.read_csv("../data/ex2.csv", names=names, index_col="message")

**Handling Missing Values (at load time):** Pandas is smart about identifying missing data (empty strings, 'NA', 'NULL'), but we can also define our own "sentinels" for missing values.

In [ ]:
# 👉 This file writes missing values in several different ways.
!cat ../data/ex5.csv

In [ ]:
# 👉 Out of the box, pandas already recognises empty fields plus 'NA' and 'NULL' as missing.
# Default behavior: reads 'NA' and 'NULL' as NaN (Not a Number/Missing)
result = pd.read_csv("../data/ex5.csv")

result

In [ ]:
# 👉 Confirm where the holes ended up.
# Verify where the missing values are detected
result.isna()

You can customize what is considered "Missing" for each column specifically.

In [ ]:
# 👉 Sometimes 'missing' is spelled differently per column. This dictionary says which
#    text counts as missing in which column. `keep_default_na=False` turns off pandas'
#    own list, so only your values count.
# Define a dictionary of sentinels
# For column 'message', treat 'NULL' and 'NA' as null
# For column 'something', treat 'two' as null
sentinels = {"message": ["NULL", "NA"], "something": ["two"]}

pd.read_csv("../data/ex5.csv", na_values=sentinels, keep_default_na=False)

**Reading Excel:** Reading Excel files works similarly, but we can specify sheet names.

In [ ]:
# 👉 Excel files can hold several sheets, so first open the workbook and see what is inside.
# Load the Excel file object
xlsx = pd.ExcelFile("../data/Resaleflatpricesbasedonregistrationdate.xlsx")

# List available sheets
xlsx.sheet_names

In [ ]:
# 👉 `.parse()` reads one named sheet into a DataFrame.
# Parse a specific sheet into a DataFrame
xlsx.parse(sheet_name="2017")

In [ ]:
# 👉 The one-line shortcut when you already know the sheet you want.
# Shortcut: Read directly without creating an ExcelFile object first
frame = pd.read_excel("../data/Resaleflatpricesbasedonregistrationdate.xlsx", sheet_name="2017")

frame

### 4.2: Writing Data (Exporting)

Once your data is cleaned, you need to save it. `to_csv` is the most common method.

In [ ]:
# 👉 Saving is the mirror of reading: `to_csv` writes the DataFrame back out to a file.
result.to_csv("../data/out.csv")

In [ ]:
# 👉 Peek at what was written. Notice the extra unnamed first column -- that is the index.
!cat ../data/out.csv

**Tip:** You usually want `index=False` to avoid saving the row numbers as a separate column.

In [ ]:
# 👉 `index=False` leaves the row labels out, which is usually what you want when the
#    index is just row numbers.
result.to_csv("../data/out.csv", index=False)

**JSON export:** the format APIs and web services speak. Same idea as `to_csv`.

In [ ]:
# 👉 `orient="records"` writes a list of one object per row, which is what most web APIs
#    expect. `indent=2` makes the file readable by humans.
clean.to_json("../data/patients_clean.json", orient="records", indent=2)

# 👉 The leading `!` runs a terminal command. `head -12` prints the first 12 lines.
!head -12 ../data/patients_clean.json

In [ ]:
# 👉 And straight back in. Dates come back as numbers or text unless you say otherwise --
#    a good reminder that CSV and JSON both forget types. Databases and pickle remember them.
pd.read_json("../data/patients_clean.json").head()

**Save our session's work.** Everything from Parts 2 and 3 lives in `clean`. Write it out
so the next notebook can start from clean data instead of repeating the whole clean-up.

In [ ]:
# 👉 `index=False` leaves out the row numbers, which carry no meaning here.
clean.to_csv("../data/patients_clean.csv", index=False)

# 👉 Verify by reading it back and running the first two moves of the ritual on it.
saved = pd.read_csv("../data/patients_clean.csv")

print(saved.shape)
saved.head()

In [ ]:
# 👉 Confirm the stray column is gone.
!cat ../data/out.csv

**Excel Export:**

In [ ]:
# 👉 Writing Excel needs a 'writer' object when you want control over sheets.
#    Always `.close()` it -- that is the step that actually flushes the file to disk.
# Option 1: Using ExcelWriter (good for multiple sheets)
writer = pd.ExcelWriter('../data/out.xlsx')

frame.to_excel(writer, 'Sheet1')

writer.close()

In [ ]:
# 👉 The one-line version for a single sheet.
# Option 2: Direct export
frame.to_excel('../data/out.xlsx')

### 4.3: Databases

Connecting to a database using `sqlalchemy`.

In [ ]:
# 👉 SQLAlchemy is the library pandas uses to talk to databases.
import sqlalchemy as sqla

In [ ]:
# 👉 `os.path.pardir` is '..' (the parent folder) and `abspath` turns it into a full path,
#    so the database file is found no matter where the notebook is launched from.
import os 

parent_dir = os.path.abspath(os.path.pardir)

In [ ]:
# 👉 A connection string says which database engine and which file. The `f'...'` prefix
#    is an f-string: whatever is inside {curly braces} gets substituted in.
# Create connection string
engine = sqla.create_engine(f'duckdb:///{parent_dir}/data/unit-1-4.db')

In [ ]:
# 👉 Give a table name and pandas reads the whole table into a DataFrame.
# Read entire table
df = pd.read_sql('resale_flat_prices_2017', engine)

In [ ]:
# 👉 Display it.
df

In [ ]:
# 👉 Or hand it real SQL and only the query result comes back. Same DataFrame either way.
# Execute raw SQL query
df = pd.read_sql("SELECT * FROM resale_flat_prices_2017", engine)

In [ ]:
# 👉 List the tables in the database. (This method is deprecated -- a warning is normal.)
engine.table_names()

> **Task:** Write a filtered DataFrame to a new database table.
> 1. Filter for flats in "YISHUN".
> 2. Write to table `yishun_flat_prices_2017`.

In [ ]:
# 👉 Filter to one town, then write those rows into a new database table.
import warnings

# Re-read the table into its own variable: `df` gets reused many times earlier in
# this notebook, so do not rely on it still holding the resale-flat data here.
resale = pd.read_sql("resale_flat_prices_2017", engine)

df_yishun = resale[resale["town"] == "YISHUN"]

# to_sql defaults to if_exists="fail", which raises ValueError if you re-run this cell.
# Downgrade that to a warning so re-running the notebook does not stop here.
try:
    df_yishun.to_sql("yishun_flat_prices_2017", engine)
except ValueError as err:
    warnings.warn(f"Skipped writing table: {err} "
                  'Pass if_exists="replace" to overwrite, or "append" to add rows.',
                  UserWarning)

df_yishun.head()

> **Final Challenge:** 
> 1. Read only flats from `BISHAN` to a new dataframe.
> 2. Write the dataframe to a new database table `bishan_flat_prices_2017`.

---

---

## 📎 Appendix — Self-Study (not covered in class)

These topics fall outside today's four learning outcomes, but they are useful and the code is
runnable. Work through them after class.

### Permutation and Random Sampling
Reordering or selecting random subsets.

In [ ]:
# 👉 A 5x7 table of the numbers 0-34 to make row shuffling easy to see.
df = pd.DataFrame(np.arange(5 * 7).reshape((5, 7)))

df

In [ ]:
# 👉 `permutation(5)` returns the numbers 0-4 in random order -- a shuffled seating plan.
# Create a random order of indices
sampler = np.random.permutation(5)

sampler

In [ ]:
# 👉 `.iloc[...]` selects rows *by position*, so the table comes back in the shuffled order.
# Reorder rows
df.iloc[sampler]

In [ ]:
# 👉 `.take()` does the same job and reads a little more clearly.
# Alternative: use .take()
df.take(sampler)

Permuting columns:

In [ ]:
# 👉 `df.shape[1]` is the number of columns, so this shuffles the column positions.
column_sampler = np.random.permutation(df.shape[1])

column_sampler

In [ ]:
# 👉 `axis=1` applies the shuffle to columns instead of rows.
df.take(column_sampler, axis=1)

**Random Sample:** Getting a random subset without manually creating a sampler.

In [ ]:
# 👉 `.sample()` skips the manual shuffle: just ask for 3 random rows.
df.sample(n=3)

In [ ]:
# 👉 `replace=True` puts each pick back in the hat, so 10 draws from 5 items is possible
#    and values can repeat. This is 'sampling with replacement'.
choices = pd.Series([5, 7, -1, 6, 4])

# Sample with replacement (items can be picked more than once)
choices.sample(n=10, replace=True)

> **Task:** Sample `df` using the parameter `frac` (percentage) instead of `n` (count).